# BRSET analysis (plan v6): run once

Implements `04_results/locked_config_v6_brset.json` exactly. This is the only notebook that reads the BRSET labels. It checks the plan, the label file and all 45 prediction files against their recorded fingerprints, then computes B1 to B5 and the quality sensitivity analysis.

**Output is aggregate only** (`04_results/brset_results_v6.json`): counts, rates, ratios and intervals. No image ids, labels or per-image predictions are written out or printed. It refuses to run a second time once the results file exists.

CPU, High-RAM; about 20 to 40 minutes.

In [2]:
# Cell 1: guards and loading
from google.colab import drive
drive.mount('/content/drive')
import os, json, glob, hashlib, itertools, datetime, warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)
import numpy as np, pandas as pd
from scipy.stats import rankdata, spearmanr, norm, multivariate_normal
from scipy.optimize import brentq
from sklearn.metrics import roc_auc_score
BASE = '/content/drive/MyDrive/stochastic_orthogonality/data/dr_ensemble_v2'; BR = '/content/drive/MyDrive/stochastic_orthogonality/data/brset'
PREDS, RESULTS, AUDIT = f'{BASE}/03_predictions', f'{BASE}/04_results', f'{BASE}/99_audit'
OUT = f'{RESULTS}/brset_results_v6.json'
assert not os.path.exists(OUT), 'Results already exist: this analysis runs once.'
def sha(p, chunk=1 << 22):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()
LOCK = f'{RESULTS}/locked_config_v6_brset.json'
IREC = json.load(open(f'{AUDIT}/brset_inference_record_v2.json')); ACC = json.load(open(f'{BR}/brset_access_record.json'))
assert sha(LOCK) == IREC['lock_v6_sha256'], 'Plan changed after inference.'
assert sha(f'{BR}/label_brset.csv') == ACC['label_file_sha256'] == IREC['access_record_label_sha256'], 'Label file changed.'
for fn, h in IREC['prediction_sha256'].items(): assert sha(f'{PREDS}/{fn}') == h, f'prediction file changed: {fn}'
print('plan, label file and 45 prediction files match their fingerprints')
L = pd.read_csv(f'{BR}/label_brset.csv', dtype=str)
cols = list(L.columns); low = {c: c.lower() for c in cols}
icdr = [c for c in cols if 'icdr' in low[c]]; pid = [c for c in cols if low[c] in ('patient_id', 'patient')]; iid = [c for c in cols if low[c] in ('image_id', 'image')]
qual = [c for c in cols if low[c] == 'quality']
if len(icdr) != 1 or len(pid) != 1 or len(iid) != 1:
    raise SystemExit(f'Plan rule: ambiguous columns, stop for a recorded amendment. Column names: {cols}')
ICDR, PID, IID = icdr[0], pid[0], iid[0]
g = pd.to_numeric(L[ICDR], errors='coerce')
coh = L[g.isin([0, 1, 2, 3, 4])].copy(); coh['grade'] = g[g.isin([0, 1, 2, 3, 4])].astype(int)
coh['image_id'] = coh[IID].astype(str).str.replace(r'\.(jpg|jpeg|png)$', '', regex=True)
SRC = [('eyepacs', m) for m in ['densenet121_binary','densenet121_5class','efficientnet_b3_binary','efficientnet_b3_5class','resnet50_binary','resnet50_5class',
       'vit_base_binary','vit_base_5class','retfound_binary','retfound_5class','retfound_adversarial']] + [('ddr', m) for m in ['densenet121_5class','efficientnet_b3_5class','resnet50_5class','retfound_5class']]
def tag(k): return '' if k == 0 else f'_seed{k}'
P = {}
for s, m in SRC:
    for k in (0, 1, 2):
        d = pd.read_csv(f'{PREDS}/brset_predictions_{s}_{m}{tag(k)}_v2.csv', dtype={'image_id': str}).set_index('image_id').predicted_probability
        P[(s, m, k)] = d
common = sorted(set(coh.image_id) & set(next(iter(P.values())).index)); coh = coh.set_index('image_id').loc[common].reset_index()
P = {key: v.reindex(coh.image_id).to_numpy() for key, v in P.items()}
assert all(np.isfinite(v).all() for v in P.values())
G = coh.grade.to_numpy(); CL = pd.factorize(coh[PID].astype(str))[0]
counts = {'images_in_label_file': int(len(L)), 'cohort_images': int(len(coh)), 'patients': int(CL.max() + 1),
          'grade_counts': {int(k): int(v) for k, v in pd.Series(G).value_counts().sort_index().items()},
          'excluded_no_valid_grade': int(len(L) - (g.isin([0, 1, 2, 3, 4])).sum()), 'icdr_column': ICDR, 'quality_column': qual[0] if qual else None}
print(json.dumps(counts, indent=1))

Mounted at /content/drive
plan, label file and 45 prediction files match their fingerprints
{
 "images_in_label_file": 16266,
 "cohort_images": 16266,
 "patients": 8524,
 "grade_counts": {
  "0": 15183,
  "1": 158,
  "2": 451,
  "3": 78,
  "4": 396
 },
 "excluded_no_valid_grade": 0,
 "icdr_column": "DR_ICDR",
 "quality_column": "quality"
}


In [3]:
# Cell 2: helpers (referral, bootstrap, combos)
def refer(s, f):
    r = np.zeros(len(s), bool); r[np.argsort(-s, kind='stable')[:int(round(f * len(s)))]] = True; return r
RNG = np.random.default_rng(20261003); NB = 1000
def ci(v): v = np.asarray(v, float); v = v[np.isfinite(v)]; return [round(float(np.percentile(v, 2.5)), 4), round(float(np.percentile(v, 97.5)), 4)] if len(v) else [None, None]
def combos(va, vb, seeds=(0, 1, 2)):
    if va == vb: return [((*va, i), (*vb, j)) for i in seeds for j in seeds if i < j]
    return [((*va, i), (*vb, j)) for i in seeds for j in seeds]
def setup(mask):
    S = {'G': G[mask], 'CL': pd.factorize(CL[mask])[0]}
    S['ref'] = S['G'] >= 2; S['sev'] = S['G'] >= 3; S['pdr'] = S['G'] == 4
    S['f'] = 1.31 * S['ref'].mean(); S['f2'] = S['ref'].mean(); n = int(mask.sum())
    S['P'] = {k: v[mask] for k, v in P.items()}; S['RK'] = {k: rankdata(v) / n for k, v in S['P'].items()}
    ncl = S['CL'].max() + 1; S['W'] = [np.bincount(RNG.integers(0, ncl, ncl), minlength=ncl)[S['CL']].astype(float) for _ in range(NB)]
    return S
ALL = setup(np.ones(len(G), bool))
CORE_V = [(s, a) for s in ('eyepacs', 'ddr') for a in ('densenet121_5class', 'efficientnet_b3_5class', 'resnet50_5class', 'retfound_5class')]
EYE_V = [v for v in SRC if v[0] == 'eyepacs']
AUC = {f'{s}_{m}{tag(k)}': round(float(roc_auc_score(ALL['ref'], ALL['P'][(s, m, k)])), 4) for s, m in SRC for k in (0, 1, 2)}
print('cohort f =', round(ALL['f'], 4), '| referable', int(ALL['ref'].sum()), '| severe', int(ALL['sev'].sum()), '| PDR', int(ALL['pdr'].sum()))

cohort f = 0.0745 | referable 925 | severe 474 | PDR 396


In [4]:
# Cell 3: B1 (primary) training-data contrast, plus the four-way relationship table
def ptype(va, vb):
    if va == vb: return 'repeat_runs'
    if va[1] == vb[1]: return 'same_design_diff_source'
    return 'diff_design_same_source' if va[0] == vb[0] else 'diff_design_diff_source'
TYPES = ['repeat_runs', 'diff_design_same_source', 'same_design_diff_source', 'diff_design_diff_source']
def b1(S, tname, fk='f'):
    tgt = S[tname]; R = {k: refer(v, S[fk]) for k, v in S['P'].items() if k[:2] in CORE_V}; VEC = {}
    for i, va in enumerate(CORE_V):
        for vb in CORE_V[i:]:
            cs = combos(va, vb); ma = mb = both = 0
            for ka, kb in cs:
                x, y = tgt & ~R[ka], tgt & ~R[kb]; ma = ma + x; mb = mb + y; both = both + (x & y)
            VEC[(va, vb)] = (ma / len(cs), mb / len(cs), both / len(cs))
    def ratio(p, w):
        ma, mb, both = (x @ w for x in VEC[p]); T = tgt @ w
        return both / (ma * mb / T) if ma * mb > 0 else np.nan
    def summ(w): return {t: float(np.nanmean([ratio(p, w) for p in VEC if ptype(*p) == t])) for t in TYPES}
    obs = summ(np.ones(len(tgt))); bs = [summ(w) for w in S['W']]
    out = {t: {'mean': round(obs[t], 3), 'ci95': ci([b[t] for b in bs])} for t in TYPES}
    for a_, b_ in (('diff_design_same_source', 'diff_design_diff_source'), ('repeat_runs', 'same_design_diff_source'), ('same_design_diff_source', 'diff_design_same_source')):
        out[f'{a_}__minus__{b_}'] = {'mean': round(obs[a_] - obs[b_], 3), 'ci95': ci([b[a_] - b[b_] for b in bs])}
    out['events'] = int(tgt.sum()); return out
B1 = {'referable': b1(ALL, 'ref'), 'severe': b1(ALL, 'sev')}
B1_PRIMARY = B1['referable']['diff_design_same_source__minus__diff_design_diff_source']
B1_SUPPORTED = B1_PRIMARY['ci95'][0] is not None and B1_PRIMARY['ci95'][0] > 0
print('B1 primary (same minus different source, different designs, referable):', B1_PRIMARY, '| supported:', B1_SUPPORTED)
print('B1b severe:', B1['severe']['diff_design_same_source__minus__diff_design_diff_source'])

B1 primary (same minus different source, different designs, referable): {'mean': 0.386, 'ci95': [0.3178, 0.4648]} | supported: True
B1b severe: {'mean': 1.035, 'ci95': [0.7619, 1.3945]}


In [5]:
# Cell 4: B2 benefit at matched volume (a: all 110 pairings of the 11 EyePACS designs; b: different designs in the core 24)
def b2(S, variants, tnames, fk='f', different_arch_only=False):
    R = {k: refer(v, S[fk]) for k, v in S['P'].items() if k[:2] in variants}; out = {}
    anc = {t: 0 for t in tnames}; ens = {t: 0 for t in tnames}; cnt = 0
    for va in variants:
        for vb in variants:
            if va == vb or (different_arch_only and va[1] == vb[1]): continue
            for ka, kb in combos(va, vb):
                e = refer(np.maximum(S['RK'][ka], S['RK'][kb]), S[fk])
                for t in tnames: y = S[t]; anc[t] = anc[t] + (y & ~R[ka]); ens[t] = ens[t] + (y & ~e)
                cnt += 1
    for t in tnames:
        red = lambda w: (anc[t] @ w - ens[t] @ w) / (anc[t] @ w) if anc[t] @ w > 0 else np.nan
        out[t] = {'relative_reduction': round(float(red(np.ones(len(S['G'])))), 4), 'ci95': ci([red(w) for w in S['W']]),
                  'deployed_misses_per_pair': round(float(anc[t].sum() / cnt), 2), 'ordered_pairs_x_runs': cnt}
    return out
B2 = {'a_eyepacs_all_110_pairings': b2(ALL, EYE_V, ('sev', 'pdr', 'ref')),
      'a_eyepacs_secondary_f_equals_prevalence': b2(ALL, EYE_V, ('sev',), fk='f2'),
      'b_core24_different_architecture': b2(ALL, CORE_V, ('sev', 'pdr', 'ref'), different_arch_only=True)}
print(json.dumps({k: v.get('sev') for k, v in B2.items()}, indent=1))

{
 "a_eyepacs_all_110_pairings": {
  "relative_reduction": 0.2469,
  "ci95": [
   0.1968,
   0.3082
  ],
  "deployed_misses_per_pair": 63.03,
  "ordered_pairs_x_runs": 990
 },
 "a_eyepacs_secondary_f_equals_prevalence": {
  "relative_reduction": 0.1679,
  "ci95": [
   0.1343,
   0.2084
  ],
  "deployed_misses_per_pair": 93.39,
  "ordered_pairs_x_runs": 990
 },
 "b_core24_different_architecture": {
  "relative_reduction": 0.2503,
  "ci95": [
   0.2116,
   0.2935
  ],
  "deployed_misses_per_pair": 114.46,
  "ordered_pairs_x_runs": 432
 }
}


In [6]:
# Cell 5: B3 decomposition, B4 sibling test, B5 miss rates by grade
def tetra(p11, pa, pb):
    if min(pa, pb) <= 0 or max(pa, pb) >= 1: return np.nan
    za, zb = norm.ppf(pa), norm.ppf(pb)
    try: return brentq(lambda r: multivariate_normal.cdf([za, zb], mean=[0, 0], cov=[[1, r], [r, 1]]) - p11, -0.999, 0.999)
    except ValueError: return np.nan
CORE = [(*v, k) for v in CORE_V for k in (0, 1, 2)]; R = {k: refer(ALL['P'][k], ALL['f']) for k in CORE}; tgt = ALL['ref']; T = tgt.sum(); rows = []
for a, b in itertools.combinations(CORE, 2):
    x, y = tgt & ~R[a], tgt & ~R[b]
    rows.append({'rho': tetra((x & y).sum() / T, x.sum() / T, y.sum() / T), 'same_source': int(a[0] == b[0]), 'same_arch': int(a[1] == b[1]), 'repeat': int(a[:2] == b[:2])})
V = pd.DataFrame(rows).dropna(); X = np.column_stack([np.ones(len(V)), V.same_source, V.same_arch, V.repeat]); beta, *_ = np.linalg.lstsq(X, V.rho.values, rcond=None)
B3 = {'floor_intercept': round(float(beta[0]), 4), 'same_training_source': round(float(beta[1]), 4), 'same_architecture': round(float(beta[2]), 4), 'repeat_runs_extra': round(float(beta[3]), 4), 'pairs': int(len(V))}
def kappa(a, b):
    po = (a == b).mean(); pa, pb = a.mean(), b.mean(); pe = pa * pb + (1 - pa) * (1 - pb); return (po - pe) / (1 - pe)
K = pd.DataFrame([{'type': ptype(a[:2], b[:2]), 'kappa': kappa(R[a], R[b])} for a, b in itertools.combinations(CORE, 2)])
thr = float(np.percentile(K.kappa[K.type.str.startswith('diff_design')], 95))
B4 = {'relative_threshold_95th_pct_diff_design': round(thr, 4), 'repeat_runs_flagged': round(float((K.kappa[K.type == 'repeat_runs'] > thr).mean()), 3),
      'same_design_diff_source_flagged': round(float((K.kappa[K.type == 'same_design_diff_source'] > thr).mean()), 3),
      'median_kappa_by_type': K.groupby('type').kappa.median().round(3).to_dict()}
B4['supported'] = B4['repeat_runs_flagged'] >= 0.80
B5 = {}
for gr in (2, 3, 4):
    y = ALL['G'] == gr; miss, resc = [], []
    for va in CORE_V:
        for vb in CORE_V:
            if va[1] == vb[1]: continue
            for ka, kb in combos(va, vb):
                e = refer(np.maximum(ALL['RK'][ka], ALL['RK'][kb]), ALL['f']); na = (y & ~R[ka]).sum()
                miss.append((y & ~R[kb]).sum() / max(y.sum(), 1))
                if na: resc.append((na - (y & ~e).sum()) / na)
    B5[f'grade_{gr}'] = {'images': int(y.sum()), 'median_per_model_miss_rate': round(float(np.median(miss)), 4), 'median_rescue_share_matched_volume': round(float(np.median(resc)), 4) if resc else None}
print('B3', B3); print('B4', B4); print('B5', B5)

B3 {'floor_intercept': 0.5698, 'same_training_source': 0.0953, 'same_architecture': 0.0775, 'repeat_runs_extra': 0.1053, 'pairs': 276}
B4 {'relative_threshold_95th_pct_diff_design': 0.6721, 'repeat_runs_flagged': 0.625, 'same_design_diff_source_flagged': 0.083, 'median_kappa_by_type': {'diff_design_diff_source': 0.446, 'diff_design_same_source': 0.531, 'repeat_runs': 0.728, 'same_design_diff_source': 0.499}, 'supported': False}
B5 {'grade_2': {'images': 451, 'median_per_model_miss_rate': 0.4268, 'median_rescue_share_matched_volume': 0.057}, 'grade_3': {'images': 78, 'median_per_model_miss_rate': 0.0577, 'median_rescue_share_matched_volume': 0.3333}, 'grade_4': {'images': 396, 'median_per_model_miss_rate': 0.2664, 'median_rescue_share_matched_volume': 0.0966}}


In [7]:
# Cell 6: quality sensitivity analysis, model AUCs, write aggregate results
SENS = None
if qual:
    q = coh[qual[0]].astype(str).str.lower()
    bad = q.str.contains('inadequ') | q.str.contains('insuf') | q.isin(['0', 'no', 'poor', 'bad'])
    if bad.any() and (~bad).sum() > 1000:
        Q = setup((~bad).to_numpy())
        SENS = {'excluded_inadequate_quality': int(bad.sum()),
                'B1_referable_primary_contrast': b1(Q, 'ref')['diff_design_same_source__minus__diff_design_diff_source'],
                'B2a_severe': b2(Q, EYE_V, ('sev',))['sev'], 'B2b_severe': b2(Q, CORE_V, ('sev',), different_arch_only=True)['sev']}
    else: SENS = {'note': 'quality field present but no images identifiable as inadequate; sensitivity analysis not applicable'}
else: SENS = {'note': 'no quality field; sensitivity analysis not applicable'}
auc_summary = {src: {'median': round(float(np.median([v for k, v in AUC.items() if k.startswith(src)])), 4),
                     'range': [round(min(v for k, v in AUC.items() if k.startswith(src)), 4), round(max(v for k, v in AUC.items() if k.startswith(src)), 4)]} for src in ('eyepacs', 'ddr')}
auc_core = {src: round(float(np.median([AUC[f'{src}_{a}{tag(k)}'] for a in ('densenet121_5class', 'efficientnet_b3_5class', 'resnet50_5class', 'retfound_5class') for k in (0, 1, 2)])), 4) for src in ('eyepacs', 'ddr')}
res = {'generated_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'plan': 'locked_config_v6_brset.json', 'plan_sha256': IREC['lock_v6_sha256'],
       'cohort': counts, 'referral_share_f': round(ALL['f'], 4), 'events': {'referable': int(ALL['ref'].sum()), 'severe': int(ALL['sev'].sum()), 'pdr': int(ALL['pdr'].sum())},
       'auc_referable_by_training_source_all_models': auc_summary, 'auc_referable_core24_median': auc_core,
       'B1_training_data': B1, 'B1_primary': B1_PRIMARY, 'B1_supported': bool(B1_SUPPORTED), 'B2_benefit': B2,
       'B3_decomposition': B3, 'B4_sibling_test': B4, 'B5_miss_rate_by_grade': B5, 'sensitivity_quality': SENS,
       'note': 'Aggregate results only. Run once under plan v6.'}
json.dump(res, open(OUT, 'w'), indent=2, default=str)
drive.flush_and_unmount(); drive.mount('/content/drive')
print('Wrote 04_results/brset_results_v6.json'); print(json.dumps({'B1_primary': B1_PRIMARY, 'B1_supported': B1_SUPPORTED, 'B2a_severe': B2['a_eyepacs_all_110_pairings']['sev'], 'B4_supported': B4['supported']}, indent=1))

Mounted at /content/drive
Wrote 04_results/brset_results_v6.json
{
 "B1_primary": {
  "mean": 0.386,
  "ci95": [
   0.3178,
   0.4648
  ]
 },
 "B1_supported": true,
 "B2a_severe": {
  "relative_reduction": 0.2469,
  "ci95": [
   0.1968,
   0.3082
  ],
  "deployed_misses_per_pair": 63.03,
  "ordered_pairs_x_runs": 990
 },
 "B4_supported": false
}
